# 02 — Làm sạch, chuẩn hóa metadata và phân tích mô tả (Giai đoạn 2)

## 1. Mục tiêu
Giai đoạn này chuyển snapshot thô của Giai đoạn 1 thành corpus trung gian (`01_data/interim/`) dùng cho gán nhãn và mô hình hóa,
đồng thời mô tả đặc điểm dữ liệu trước khi có nhãn. Shopee (corpus chính) và Lazada (External Test) được xử lý song song nhưng
lưu tách biệt. Dữ liệu thô không bị chỉnh sửa.

| Đầu vào (read-only) | Đầu ra |
|---|---|
| `raw/shopee/shopee_reviews_stratified_T0_2026-10-06.jsonl` | `interim/shopee_clean_T0_2026-10-06.jsonl` (+ `_dropped`) |
| `raw/lazada/lazada_reviews_T0_2026-10-06_r2.jsonl` | `interim/lazada_clean_T0_2026-10-06.jsonl` (+ `_dropped`) |

## 2. Thiết kế xử lý
| Bước | Quy tắc |
|---|---|
| Khử trùng | Theo `review_id` (kiểm lại sau bước gộp ở Giai đoạn 1) |
| Cửa sổ nghiên cứu | Giữ review có `review_date` trong [2025-10-06 00:00, 2026-10-06 23:59:59] (+07:00) (amendment v1.5) |
| Ẩn danh hóa | Áp lại luật scrub hiện hành cho `text` và `reply_content` (luật có thể được bổ sung sau QA) |
| Chuẩn hóa văn bản | NFC, bỏ ký tự điều khiển/định dạng (giữ ZWJ của emoji), gộp khoảng trắng; không chuyển chữ thường, không bỏ emoji, không sửa chính tả — phù hợp nguyên tắc làm sạch tối thiểu (ĐC §7.1) |
| Lọc | Loại `text_clean` rỗng hoặc không chứa chữ cái (chỉ emoji/ký hiệu/số); các dòng bị loại được lưu kèm lý do |
| Đánh dấu (không xóa) | `near_duplicate` (text_clean trùng y hệt với review khác), `non_vi` / `lang_und` (phát hiện ngôn ngữ bằng luật) |

Schema thống nhất: `review_id, source_app, text, text_clean, rating, review_date, app_version, thumbs_up, reply_content,
replied_at, lang, flags`. Trường `text` giữ văn bản gốc đã ẩn danh hóa để truy vết; `text_clean` là bản chuẩn hóa.
Logic nằm trong `src/preprocessing/clean.py` và `src/preprocessing/eda.py`.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath(os.path.join('..', '02_src')))
from src.utils.kaggle_paths import input_dir, working_dir
from src.utils.seeds import set_seed
from src.preprocessing.clean import (clean_app, load_jsonl, pii_qa, write_manifest,
                                     WINDOW_START, WINDOW_END)
from src.preprocessing import eda
set_seed(42)
TAG = 'T0_2026-10-06'
RAW, INTERIM, MANIFESTS = input_dir('raw'), input_dir('interim'), input_dir('manifests')
EDA_DIR = working_dir('eda')
FIG_DIR = working_dir(os.path.join('eda', 'figures'))
SOURCES = {
    'shopee': os.path.join(RAW, 'shopee', f'shopee_reviews_stratified_{TAG}.jsonl'),
    'lazada': os.path.join(RAW, 'lazada', f'lazada_reviews_{TAG}_r2.jsonl'),
}
print('Cửa sổ nghiên cứu:', WINDOW_START.isoformat(), '→', WINDOW_END.isoformat())

Cửa sổ nghiên cứu: 2025-10-06T00:00:00+07:00 → 2026-10-06T23:59:59+07:00


## 3. Làm sạch và cắt cửa sổ thời gian

In [2]:
runs, records = {}, {}
for app, raw_path in SOURCES.items():
    out = os.path.join(INTERIM, f'{app}_clean_{TAG}.jsonl')
    dropped = os.path.join(INTERIM, f'{app}_dropped_{TAG}.jsonl')
    runs[app] = clean_app(raw_path, app, out, dropped)
    records[app] = load_jsonl(out)

labels = [('Raw', 'raw'), ('Trùng review_id', 'duplicate_review_id'),
          ('Ngoài cửa sổ (trước 2025-10-06)', 'outside_window_before'),
          ('Ngoài cửa sổ (sau 2026-10-06 23:59)', 'outside_window_after'),
          ('Sau cắt cửa sổ', 'after_window'), ('Loại: text rỗng', 'drop_empty'),
          ('Loại: không có chữ cái', 'drop_no_letters'), ('Sau lọc (interim)', 'after_filter'),
          ('Thay PII bổ sung ở interim', 'pii_replacements_at_interim')]
print(f"{'Bước':40s}{'Shopee':>10s}{'Lazada':>10s}")
for label, k in labels:
    print(f"{label:40s}{runs['shopee']['steps'][k]:>10,}{runs['lazada']['steps'][k]:>10,}")

Bước                                        Shopee    Lazada
Raw                                         57,200     3,600
Trùng review_id                                  0         0
Ngoài cửa sổ (trước 2025-10-06)                831        94
Ngoài cửa sổ (sau 2026-10-06 23:59)              0         0
Sau cắt cửa sổ                              56,369     3,506
Loại: text rỗng                                  0         0
Loại: không có chữ cái                         319        12
Sau lọc (interim)                           56,050     3,494
Thay PII bổ sung ở interim                       5     1,105


## 4. Manifest của corpus trung gian
Manifest ghi số lượng qua từng bước, khoảng ngày, checksum của tệp nguồn và tệp đầu ra. Khoảng ngày của interim được kiểm
tra nằm trọn trong cửa sổ nghiên cứu.

In [3]:
from datetime import datetime
manifest_path = os.path.join(MANIFESTS, f'gd2_interim_manifest_{TAG}.json')
manifest = write_manifest(list(runs.values()), manifest_path)
for app, r in runs.items():
    lo, hi = datetime.fromisoformat(r['min_review_date']), datetime.fromisoformat(r['max_review_date'])
    assert WINDOW_START <= lo and hi <= WINDOW_END, f'{app}: ngoài cửa sổ'
    print(f"{app:7s} min={lo.astimezone(WINDOW_START.tzinfo)} max={hi.astimezone(WINDOW_START.tzinfo)} "
          f"| raw sha256={r['raw_sha256'][:12]}… | interim sha256={r['out_sha256'][:12]}…")
print('Manifest:', manifest_path)

shopee  min=2025-10-06 00:03:56+07:00 max=2026-10-06 07:18:03+07:00 | raw sha256=d8115c1fd7d9… | interim sha256=42d1e4811fc5…
lazada  min=2025-10-06 06:09:42+07:00 max=2026-10-06 05:45:41+07:00 | raw sha256=c7c882890de9… | interim sha256=1cd44b596fff…
Manifest: C:\Workspace\kltn-vn-app-review-requirements\01_data\manifests\gd2_interim_manifest_T0_2026-10-06.json


## 5. Kiểm tra chất lượng ẩn danh hóa
Mỗi ứng dụng được lấy ngẫu nhiên (seed 42) 50 `reply_content` và 30 `text` để soi trực tiếp. Song song, một bộ regex phụ được
chạy trên toàn corpus để tìm dấu hiệu sót: chuỗi 2–3 từ viết hoa đầu từ sau "bạn/anh/chị/em", chuỗi từ 9 chữ số trở lên,
chuỗi có ký tự `@`.

In [4]:
qa = {}
for app in SOURCES:
    qa[app] = pii_qa(records[app], n_reply=50, n_text=30, seed=42)
    q = qa[app]
    print(f'===== {app}: mẫu reply={q["n_sample_reply"]}, mẫu text={q["n_sample_text"]}, '
          f'nghi sót trong mẫu={q["n_suspects_in_sample"]}')
    print('Nghi sót trên toàn corpus:', q['suspects_full_corpus'] or 'không có')
    for kind, ex in q['suspect_examples'].items():
        print(f'  {kind}:', [frag for _, frag in ex])

===== shopee: mẫu reply=50, mẫu text=30, nghi sót trong mẫu=0
Nghi sót trên toàn corpus: {'reply_content:long_digits': 3, 'text:long_digits': 8, 'text:title_after_pronoun': 10}
  reply_content:long_digits: ['1441554913', '1441554913', '1441554913']
  text:long_digits: ['01-16-27-34-48-54', '066991025085', '61 7444444441', '100000000', '1000000000000000', '1000000000.000000', '1000000000', '10000000000000000']
  text:title_after_pronoun: ['Anh Không Làm Được', 'Em Cuối Chiều Em', 'Anh Vực Sâu Tối', 'Em Chẳng Màng Chỉ', 'Anh Ngỡ Ngàng Em', 'Anh Đã Thấy Đôi', 'Em Mỉm Cười Bên', 'Anh Giấu Đi Lệ']
===== lazada: mẫu reply=50, mẫu text=30, nghi sót trong mẫu=0
Nghi sót trên toàn corpus: không có


In [5]:
# Mẫu soi trực tiếp (cắt 160 ký tự mỗi dòng)
for app in SOURCES:
    print(f'===== {app} — reply_content')
    for rid, s in qa[app]['sample_reply']:
        print(f'  {rid[:8]} | {s[:160]}')
    print(f'===== {app} — text')
    for rid, s in qa[app]['sample_text']:
        print(f'  {rid[:8]} | {s[:160]}')

===== shopee — reply_content
  4123db46 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  09f05b73 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  b8c0ae57 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  f0dfede2 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  e0e8b8af | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụng và chất lượng dịch vụ hơn nữa. Để có thể hỗ trợ bạn nhanh chóng và hi
  3756aba0 | Chào bạn, cảm ơn bạn đã phản hồi. Shopee xin ghi nhận ý kiến của bạn để cải thiện ứng dụ

Kết quả QA. Bước kiểm tra được lặp lại cho tới khi không còn sót rõ ràng, qua hai đợt bổ sung luật:

1. Regex phụ trên toàn corpus phát hiện tên còn sót trong `reply_content`: (i) Shopee — tên hiển thị sau "Chào bạn" bắt đầu bằng
   từ có thể là đại từ hoặc tính từ ("Anh MvP", "Nha Thanh", "Vui Dang"); (ii) Lazada — phản hồi mở đầu trực tiếp bằng tên trước
   ", cảm ơn" và một phản hồi tiếng Na Uy ("Hei <tên>!").
2. Đọc trực tiếp mẫu 50 phản hồi Lazada phát hiện thêm 3/50 tên còn sót mà regex phụ không bắt được: tên đứng đầu trước
   ", chúng tôi", ", mình" hoặc câu tiếng Anh ("<tên>, we appreciate …"). Luật "tên đứng đầu" được mở rộng cho các từ nối này.

Các luật được bổ sung trong `scrub_pii` (`src/collect/google_play.py`, chỉ áp cho phản hồi của sàn) và áp lại ở bước interim;
mọi đoạn bị thay ở mỗi đợt đều được rà soát và là tên hiển thị (ngoại lệ vô hại: lời xưng hô tiếng Indonesia "Kakak").
Các ca regex phụ còn lại của Shopee là dương tính giả (số tiền, mã vận đơn `SPXVN…`, mã sự kiện trong URL, lời bài hát viết hoa
từng chữ). Tệp raw của Giai đoạn 1 giữ nguyên nên vẫn chứa các tên bị sót; corpus dùng cho các giai đoạn sau là interim.

## 6. Phân tích mô tả
Phân tích chỉ mô tả dữ liệu; chưa có nhãn nên không có ước lượng tỷ lệ actionable hay BUG/FEATURE/UX ở giai đoạn này.
Thời gian được quy về giờ Việt Nam (+07:00).

In [6]:
dfs = {app: eda.to_frame(records[app], app) for app in SOURCES}
summaries = {app: eda.summarize(dfs[app], runs[app]['steps']['drop_no_letters'], runs[app]['steps']['after_window'])
             for app in SOURCES}
for app, s in summaries.items():
    print(f'===== {app}')
    print(json.dumps(s, ensure_ascii=False, indent=1, default=str))

===== shopee
{
 "n_reviews": 56050,
 "date_min": "2025-10-06 00:03:56+07:00",
 "date_max": "2026-10-06 07:18:03+07:00",
 "rating_pct": {
  "1": 39.9,
  "2": 3.4,
  "3": 3.2,
  "4": 5.0,
  "5": 48.5
 },
 "words_q25_q50_q75_q90": [
  2.0,
  6.0,
  16.0,
  32.0
 ],
 "chars_median": 27.0,
 "short_le2_words_pct": 28.6,
 "app_version_missing_pct": 26.3,
 "top_versions": {
  "3.65.22": 2729,
  "3.61.28": 2416,
  "3.81.32": 2254,
  "3.77.25": 2107,
  "3.66.26": 1828
 },
 "reply_pct": 60.3,
 "lang_pct": {
  "vi": 90.9,
  "und": 8.8,
  "non_vi": 0.3
 },
 "near_dup_pct": 30.1,
 "no_letters_dropped_pct": 0.57
}
===== lazada
{
 "n_reviews": 3494,
 "date_min": "2025-10-06 06:09:42+07:00",
 "date_max": "2026-10-06 05:45:41+07:00",
 "rating_pct": {
  "1": 60.7,
  "2": 3.5,
  "3": 3.8,
  "4": 3.7,
  "5": 28.3
 },
 "words_q25_q50_q75_q90": [
  4.0,
  11.0,
  25.0,
  47.0
 ],
 "chars_median": 47.0,
 "short_le2_words_pct": 19.3,
 "app_version_missing_pct": 34.0,
 "top_versions": {
  "7.88.0": 221,
  "7.89

In [7]:
figures = eda.all_figures(dfs, FIG_DIR)
for f in figures:
    print(f)

C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_rating_distribution.png
C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_length_words.png
C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_monthly_volume.png
C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_daily_tail.png
C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\figures\eda_app_version_top.png


## 7. Nhận xét và báo cáo
Nhận xét dưới đây được viết sau khi đọc các số liệu và hình ở mục 6; báo cáo đầy đủ (bảng đối soát, bảng tóm tắt, QA,
hạn chế, danh sách hình) được sinh vào `05_experiments/eda/eda_report.md`.

In [8]:
NOTES = {
    'shopee': (
        'Corpus gồm 56.050 review trong cửa sổ 12 tháng. Phân bố mức đánh giá phân cực: 1★ và 5★ chiếm gần 90%, '
        'các mức 2–4★ chỉ khoảng 11–12%. Số review mỗi tháng dao động khoảng 3.500–6.300, cao nhất vào 01/2026. Review ngắn chiếm tỷ trọng lớn (trung vị 6 từ; khoảng 29% có không quá 2 từ) '
        'và khoảng 30% review trùng y hệt nội dung với review khác, chủ yếu là các câu chung chung như "ok", "tốt", '
        '"rất tốt". Khoảng 60% review có phản hồi của sàn; 26,3% thiếu app_version.'),
    'lazada': (
        'Corpus gồm 3.494 review. Phân bố mức đánh giá nghiêng về tiêu cực hơn Shopee (1★ khoảng 61%, 5★ khoảng 28%). '
        'Số review mỗi tháng dao động khoảng 190–520, cao nhất vào 12/2025. Review dài hơn (trung vị 11 từ; khoảng 19% có không quá 2 từ) và tỷ lệ trùng nội dung thấp hơn (khoảng 15%). '
        'Toàn bộ review đều có phản hồi của sàn, và phần lớn phản hồi gọi tên người dùng — đây là nguồn chính của các '
        'lượt ẩn danh hóa. 34,0% thiếu app_version.'),
    'qa_action': (
        'Hai đợt bổ sung luật scrub cho phản hồi của sàn. Đợt 1 (từ regex phụ): tên sau "Chào bạn" bắt đầu bằng từ có thể '
        'là đại từ/tính từ; phản hồi mở đầu trực tiếp bằng tên trước ", cảm ơn". Đợt 2 (từ đọc trực tiếp mẫu, 3/50 phản hồi '
        'Lazada còn sót): tên đứng đầu trước ", chúng tôi", ", mình", hoặc câu tiếng Anh. Sau hai đợt, mẫu soi lại và regex phụ '
        'không còn ca sót ở Lazada; các ca regex phụ còn lại của Shopee đã kiểm tra thủ công và đều là dương tính giả. '
        'Tệp raw giữ nguyên (bất biến, không đưa vào Git); các giai đoạn sau chỉ dùng interim.'),
}
report = eda.write_report(os.path.join(EDA_DIR, 'eda_report.md'), summaries,
                          {a: runs[a]['steps'] for a in SOURCES}, qa, figures, NOTES)
print('Báo cáo:', report)

Báo cáo: C:\Workspace\kltn-vn-app-review-requirements\05_experiments\eda\eda_report.md


## 8. Đánh giá hoàn thành Giai đoạn 2

In [9]:
print('Đối soát số lượng (raw → sau cắt cửa sổ → sau lọc):')
for app, r in runs.items():
    s = r['steps']
    print(f"  {app:7s} {s['raw']:,} → {s['after_window']:,} (ngoài cửa sổ: {s['outside_window_before'] + s['outside_window_after']:,}) "
          f"→ {s['after_filter']:,} (rỗng: {s['drop_empty']}, không có chữ cái: {s['drop_no_letters']})")
print('Khoảng ngày interim nằm trong cửa sổ: có (đã kiểm ở mục 4)')
print('QA ẩn danh hóa:')
for app, q in qa.items():
    print(f"  {app:7s} mẫu soi: {q['n_sample_reply']} reply + {q['n_sample_text']} text | nghi sót trong mẫu: "
          f"{q['n_suspects_in_sample']} | nghi sót toàn corpus (regex phụ): {sum(q['suspects_full_corpus'].values())}")
print('Hình đã lưu:')
for f in figures:
    print('  ', os.path.relpath(f, os.path.join('..')))

Đối soát số lượng (raw → sau cắt cửa sổ → sau lọc):
  shopee  57,200 → 56,369 (ngoài cửa sổ: 831) → 56,050 (rỗng: 0, không có chữ cái: 319)
  lazada  3,600 → 3,506 (ngoài cửa sổ: 94) → 3,494 (rỗng: 0, không có chữ cái: 12)
Khoảng ngày interim nằm trong cửa sổ: có (đã kiểm ở mục 4)
QA ẩn danh hóa:
  shopee  mẫu soi: 50 reply + 30 text | nghi sót trong mẫu: 0 | nghi sót toàn corpus (regex phụ): 21
  lazada  mẫu soi: 50 reply + 30 text | nghi sót trong mẫu: 0 | nghi sót toàn corpus (regex phụ): 0
Hình đã lưu:
   05_experiments\eda\figures\eda_rating_distribution.png
   05_experiments\eda\figures\eda_length_words.png
   05_experiments\eda\figures\eda_monthly_volume.png
   05_experiments\eda\figures\eda_daily_tail.png
   05_experiments\eda\figures\eda_app_version_top.png


## 9. Hạn chế
- Ngày 06/10/2026 chỉ có một phần (review mới nhất lúc 07:18 giờ Việt Nam) do Google Play hiển thị review trễ khoảng một ngày;
  dữ liệu không được bù. Ảnh hưởng nhẹ tới cửa sổ last_30 của phân tích thời gian.
- Ngôn ngữ được xác định bằng luật (dấu tiếng Việt, chữ ngoài hệ Latin, từ điển tiếng Việt không dấu); review rất ngắn không dấu
  được gắn `und`. Review không phải tiếng Việt mới chỉ được đánh dấu, chưa bị loại.
- `near_duplicate` chỉ bắt trùng y hệt sau chuẩn hóa; các biến thể gần giống (khác dấu câu, chữ hoa) không được gộp.
- Ẩn danh hóa dựa trên luật; QA mẫu không loại trừ hoàn toàn khả năng sót ở vị trí bất thường.